# Saúde — Indicadores de Diabetes: EDA + Mineração + Apoio à Decisão

**Fonte:** `alexteboul/diabetes-health-indicators-dataset` (BRFSS 2015, Kaggle).
Comando real para baixar os dados (requer conta Kaggle + `kaggle.json` configurado):

```bash
kaggle datasets download alexteboul/diabetes-health-indicators-dataset -p data --unzip
```

> **Dados reais:** `data/diabetes_binary_health_indicators_BRFSS2015.csv`
> (BRFSS 2015 original desbalanceado — n=253.680, prevalência ≈ 0,14).
> EDA, KMeans e Apriori rodam na prevalência real; só a árvore usa undersampling
> balanceado só no treino (documentado na §2) — o teste mantém a prevalência
> real ≈ 0,14 (métrica em teste balanceado seria otimista). O fallback sintético
> (seed 42, n=5000, 20 colunas — sem Education/Income, fallback aproximado)
> permanece apenas como último recurso se `data/` estiver ausente.

Blocos: entendimento → preparação (balanceamento documentado, encoding binário) → EDA →
mineração (KMeans k=3, árvore de decisão + matriz de confusão, Apriori) → decisão (regra de triagem + prevenções).

In [1]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display
from src.common import load_csv, save_fig, df_to_html_table
from src.mining import run_kmeans, run_tree, confusion_summary
RNG = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 100
print("imports ok")

imports ok


## 1. Entendimento dos dados

Alvo: `Diabetes_binary` (0 = sem diabetes, 1 = pré/diabetes). Fatores: pressão alta (`HighBP`),
colesterol alto (`HighChol`), IMC (`BMI`), idade em 13 faixas (`Age`), atividade física (`PhysActivity`),
tabagismo, AVC, doença cardíaca, saúde geral (`GenHlth` 1–5), saúde mental/física (dias ruins em 30),
dificuldade de locomoção (`DiffWalk`), sexo e hábitos.

In [2]:
# Original desbalanceado primeiro (prevalência real ~0,14); 5050split só como alternativa.
DATA = Path("..") / "data"
CANDIDATES = [
    str(DATA / "diabetes_binary_health_indicators_BRFSS2015.csv"),
    str(DATA / "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"),
]
df = None
is_real = False
for c in CANDIDATES:
    try:
        df = load_csv(c)
        print("Carregado:", c)
        is_real = True
        break
    except FileNotFoundError:
        continue
if df is None:
    is_real = False
    print("Sem data/ local -> fallback SINTÉTICO realista (seed 42, n=5000).")
    print("Comando real: kaggle datasets download alexteboul/diabetes-health-indicators-dataset -p data --unzip")
    n = 5000
    age = RNG.integers(1, 14, n)
    highbp = (RNG.random(n) < 0.25 + 0.03 * age).astype(int)
    highchol = (RNG.random(n) < 0.25 + 0.025 * age).astype(int)
    cholcheck = (RNG.random(n) < 0.95).astype(int)
    bmi = np.clip(RNG.normal(28, 6, n), 12, 60).round(1)
    smoker = (RNG.random(n) < 0.44).astype(int)
    stroke = (RNG.random(n) < 0.04 + 0.004 * age).astype(int)
    heart = (RNG.random(n) < 0.08 + 0.006 * age).astype(int)
    physact = (RNG.random(n) < 0.75 - 0.015 * age).astype(int)
    fruits = (RNG.random(n) < 0.63).astype(int)
    veggies = (RNG.random(n) < 0.81).astype(int)
    hvyalc = (RNG.random(n) < 0.06).astype(int)
    anyhc = (RNG.random(n) < 0.89).astype(int)
    nodoc = (RNG.random(n) < 0.11).astype(int)
    genhlth = np.clip(RNG.integers(1, 6, n) + (RNG.random(n) < 0.2).astype(int), 1, 5)
    menthlth = np.clip(RNG.negative_binomial(2, 0.4, n), 0, 30)
    physhlth = np.clip(RNG.negative_binomial(2, 0.35, n), 0, 30)
    diffwalk = ((physhlth > 10) | (RNG.random(n) < 0.12 + 0.01 * age)).astype(int)
    sex = RNG.integers(0, 2, n)
    logit = (-3.2 + 0.9 * highbp + 0.8 * highchol + 0.06 * (bmi - 25)
             + 0.12 * age - 0.5 * physact + 0.25 * (genhlth - 3)
             + 0.4 * heart + 0.3 * stroke)
    p = 1 / (1 + np.exp(-logit))
    diab = (RNG.random(n) < p).astype(int)
    df = pd.DataFrame({
        "Diabetes_binary": diab, "HighBP": highbp, "HighChol": highchol,
        "CholCheck": cholcheck, "BMI": bmi, "Smoker": smoker, "Stroke": stroke,
        "HeartDiseaseorAttack": heart, "PhysActivity": physact, "Fruits": fruits,
        "Veggies": veggies, "HvyAlcoholConsump": hvyalc, "AnyHealthcare": anyhc,
        "NoDocbcCost": nodoc, "GenHlth": genhlth, "MentHlth": menthlth,
        "PhysHlth": physhlth, "DiffWalk": diffwalk, "Sex": sex, "Age": age,
    })
print("shape:", df.shape)
print("prevalência:", round(float(df['Diabetes_binary'].mean()), 4))
display(HTML(df_to_html_table(df, 10)))
print("is_real:", is_real)

Carregado: ../data/diabetes_binary_health_indicators_BRFSS2015.csv
shape: (253680, 22)
prevalência: 0.1393


Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,6.0,8.0
0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,14.0,0.0,0.0,9.0,6.0,7.0
0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,0.0,11.0,4.0,4.0
1.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0
0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,8.0,4.0,3.0


is_real: True


## 2. Preparação — balanceamento documentado e encoding binário

- **Encoding:** o dataset original já é binário (0/1) nas colunas indicadoras; `GenHlth` (1–5),
`Age` (1–13) e contagens de dias são mantidos como ordinais/numéricos — nenhum one-hot necessário.
- **Balanceamento (documentado):** a prevalência é minoritária (~14% no original desbalanceado). O split 75/25
estratificado é feito **primeiro, na base real**; o undersampling aleatório da classe majoritária (seed 42) é
aplicado **só no treino**. KMeans e Apriori usam a base cheia (distribuição real preservada); o teste mantém a
prevalência real ≈ 0,14, então acurácia/PPV/NPV são medidos em prevalência real.

In [3]:
print("Distribuição original (proporção):")
print(df["Diabetes_binary"].value_counts(normalize=True).round(3).to_dict())
n0 = int((df["Diabetes_binary"] == 0).sum())
n1 = int((df["Diabetes_binary"] == 1).sum())
bal = pd.DataFrame({"classe": [0, 1], "n_base": [n0, n1],
                    "nota": ["majoritária: subamostrada à minoria só no treino (§5)",
                             "minoritária: integral no treino; contagens exatas em §5"]})
display(HTML(df_to_html_table(bal)))
print("EDA/KMeans/Apriori usam o df real completo; o undersampling é aplicado só no treino da árvore (§5),")
print("onde o treino balanceado tem 2×(minoria no treino) linhas — contagens exatas impressas em §5.")

Distribuição original (proporção):
{0.0: 0.861, 1.0: 0.139}


classe,n_base,nota
0,218334,majoritária: subamostrada à minoria só no treino (§5)
1,35346,minoritária: integral no treino; contagens exatas em §5


EDA/KMeans/Apriori usam o df real completo; o undersampling é aplicado só no treino da árvore (§5),
onde o treino balanceado tem 2×(minoria no treino) linhas — contagens exatas impressas em §5.


## 3. EDA — prevalência por idade, pressão, colesterol e atividade + correlações

Quatro leituras de prevalência orientam a triagem; o heatmap mostra quais fatores andam juntos.
**Legenda das faixas etárias** — códigos `Age` do BRFSS 2015 (faixas de 5 anos):

| Faixa | Idade | Faixa | Idade |
|---|---|---|---|
| F1 | 18–24 | F8 | 55–59 |
| F2 | 25–29 | F9 | 60–64 |
| F3 | 30–34 | F10 | 65–69 |
| F4 | 35–39 | **F11** | **70–74** |
| F5 | 40–44 | F12 | 75–79 |
| F6 | 45–49 | F13 | 80+ |
| F7 | 50–54 |  |  |

Pico em **F11 (70–74)** com 0,22; a regra de triagem "faixa ≥ 9" = **60+**.


In [4]:
def prev_bar(col, labels, fname, title, xlabel):
    g = df.groupby(col)["Diabetes_binary"].mean()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([labels[i] for i in g.index], g.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Prevalência de diabetes")
    for i, v in enumerate(g.values):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    url = save_fig(fig, fname)
    plt.close(fig)
    display(HTML(f"<img src='{url}' alt='{title}'>"))


prev_bar("Age", {i: f"F{i}" for i in range(1, 14)},
         "diabetes_prev_idade.png", "Prevalência de diabetes por faixa etária", "Faixa etária (1–13)")
prev_bar("HighBP", {0: "Sem pressao alta", 1: "Com pressao alta"},
         "diabetes_prev_pressao.png", "Prevalência de diabetes por pressão arterial", "Pressão alta")
prev_bar("HighChol", {0: "Sem colesterol alto", 1: "Com colesterol alto"},
         "diabetes_prev_colesterol.png", "Prevalência de diabetes por colesterol", "Colesterol alto")
prev_bar("PhysActivity", {0: "Sedentário", 1: "Ativo"},
         "diabetes_prev_atividade.png", "Prevalência de diabetes por atividade física", "Atividade física")

In [5]:
cols = ["Diabetes_binary", "HighBP", "HighChol", "BMI", "Smoker", "Stroke",
        "HeartDiseaseorAttack", "PhysActivity", "GenHlth", "PhysHlth",
        "DiffWalk", "Age"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[cols].corr().round(2), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlação entre fatores e diabetes")
fig.tight_layout()
url = save_fig(fig, "diabetes_correlacao.png")
plt.close(fig)
display(HTML(f"<img src='{url}' alt='Correlacoes'>"))


## 4. Mineração com KMeans (k=3, perfis de risco)

KMeans com k=3 sobre atributos padronizados (IMC, idade, saúde geral, dias ruins,
pressão e colesterol) para descobrir 3 perfis de risco na população.

In [6]:
from sklearn.preprocessing import StandardScaler
feats = ["BMI", "Age", "GenHlth", "PhysHlth", "MentHlth", "HighBP", "HighChol"]
Xs = StandardScaler().fit_transform(df[feats])
labels, km = run_kmeans(pd.DataFrame(Xs, columns=feats), k=3, seed=42)
df["cluster"] = labels
prof = df.groupby("cluster")[feats + ["Diabetes_binary"]].mean().round(3).reset_index()
prof = prof.sort_values("Diabetes_binary").reset_index(drop=True)
display(HTML(df_to_html_table(prof)))
print("KMeans k=3 — tamanho dos clusters:", pd.Series(labels).value_counts().sort_index().to_dict())

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df["BMI"], df["Age"], c=df["cluster"], cmap="viridis", alpha=0.4, s=8)
ax.set_title("KMeans k=3 — clusters no plano IMC x idade")
ax.set_xlabel("BMI")
ax.set_ylabel("Faixa etária")
fig.colorbar(sc, ax=ax, label="cluster")
fig.tight_layout()
url = save_fig(fig, "diabetes_kmeans.png")
plt.close(fig)
display(HTML(f"<img src='{url}' alt='Clusters KMeans'>"))

cluster,BMI,Age,GenHlth,PhysHlth,MentHlth,HighBP,HighChol,Diabetes_binary
1,26.677,6.820,2.026,1.134,1.776,0.017,0.248,0.043
2,29.886,9.545,2.652,1.660,1.175,0.940,0.612,0.220
0,30.782,8.584,3.937,22.421,13.604,0.624,0.588,0.287


KMeans k=3 — tamanho dos clusters: {0: 34806, 1: 128569, 2: 90305}


## 5. Árvore de decisão + matriz de confusão

A árvore prevê `Diabetes_binary` com split 75/25 estratificado feito primeiro na base real;
o undersampling balanceia **só o treino** e o teste mantém a prevalência real ≈ 0,14.
A matriz de confusão e a acurácia resumem o desempenho em prevalência real
(métrica em teste balanceado seria otimista); a importância revela o top-5 de fatores.

In [7]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Diabetes_binary", "cluster"], errors="ignore")
y = df["Diabetes_binary"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
neg = Xtr[ytr == 0].sample(int((ytr == 1).sum()), random_state=42)  # undersample SÓ treino
pos = Xtr[ytr == 1]
Xtr_b = pd.concat([neg, pos]).sample(frac=1, random_state=42)
ytr_b = ytr.loc[Xtr_b.index]
print(f"treino balanceado: {Xtr_b.shape}, holdout prevalência: {float(yte.mean()):.4f}")
tree = run_tree(Xtr_b, ytr_b)
pred = tree.predict(Xte)  # Xte com prevalência real ~0,14
s = confusion_summary(yte, pred)
print("Acurácia:", round(s["accuracy"], 3))
print("Matriz de confusão [[VN, FP], [FN, VP]]:", s["confusion_matrix"])
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
top5 = imp.head(5).reset_index()
top5.columns = ["fator", "importancia"]
top5["importancia"] = top5["importancia"].round(4)
display(HTML(df_to_html_table(top5)))
print("Top-5 fatores:", ", ".join(top5["fator"]))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(s["confusion_matrix"], annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Matriz de confusão — árvore de decisão")
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
fig.tight_layout()
url = save_fig(fig, "diabetes_confusao.png")
plt.close(fig)
display(HTML(f"<img src='{url}' alt='Matriz de confusao'>"))

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top5["fator"][::-1], top5["importancia"][::-1])
ax.set_title("Top-5 fatores (importância da árvore)")
ax.set_xlabel("Importância")
fig.tight_layout()
url_imp = save_fig(fig, "diabetes_importancia.png")
plt.close(fig)
display(HTML(f"<img src='{url_imp}' alt='Top fatores'>"))

treino balanceado: (53018, 21), holdout prevalência: 0.1393
Acurácia: 0.659
Matriz de confusão [[VN, FP], [FN, VP]]: [[36045, 18538], [3115, 5722]]


fator,importancia
HighBP,0.1480
BMI,0.1380
Age,0.1039
GenHlth,0.1003
Income,0.0891


Top-5 fatores: HighBP, BMI, Age, GenHlth, Income


In [8]:
print(f"prevalência holdout: {float(yte.mean()):.4f} (base: {float(y.mean()):.4f})")
assert abs(float(yte.mean()) - float(y.mean())) < 0.02

prevalência holdout: 0.1393 (base: 0.1393)


## 6. Regras de associação (discretização + Apriori com mlxtend)

Fatores já binários dispensam discretização fina (variáveis contínuas como IMC/idade foram
excluídas da cesta). O Apriori (suporte ≥ 0.10, lift > 1.1) encontra co-ocorrências que viram regras.

In [9]:
from mlxtend.frequent_patterns import apriori, association_rules
basket = pd.DataFrame({
    "HighBP": df["HighBP"], "HighChol": df["HighChol"], "Smoker": df["Smoker"],
    "NoActivity": 1 - df["PhysActivity"], "Heart": df["HeartDiseaseorAttack"],
    "DiffWalk": df["DiffWalk"], "Diabetes": df["Diabetes_binary"],
})
freq = apriori(basket.astype(bool), min_support=0.10, use_colnames=True)
rules = association_rules(freq, metric="lift", min_threshold=1.1)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
top = rules.head(10)[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
top["antecedents"] = top["antecedents"].astype(str)
top["consequents"] = top["consequents"].astype(str)
top[["support", "confidence", "lift"]] = top[["support", "confidence", "lift"]].round(3)
print(f"{len(freq)} itemsets frequentes, {len(rules)} regras (lift>1.1)")
display(HTML(df_to_html_table(top, 10)))

15 itemsets frequentes, 22 regras (lift>1.1)


antecedents,consequents,support,confidence,lift
frozenset({'Diabetes'}),frozenset({'HighBP'}),0.105,0.753,1.754
frozenset({'HighBP'}),frozenset({'Diabetes'}),0.105,0.244,1.754
frozenset({'DiffWalk'}),frozenset({'HighBP'}),0.114,0.675,1.574
frozenset({'HighBP'}),frozenset({'DiffWalk'}),0.114,0.265,1.574
"frozenset({'HighChol', 'Smoker'})",frozenset({'HighBP'}),0.134,0.637,1.484
frozenset({'HighBP'}),"frozenset({'HighChol', 'Smoker'})",0.134,0.312,1.484
frozenset({'HighChol'}),"frozenset({'HighBP', 'Smoker'})",0.134,0.316,1.476
"frozenset({'HighBP', 'Smoker'})",frozenset({'HighChol'}),0.134,0.626,1.476
frozenset({'HighBP'}),frozenset({'HighChol'}),0.255,0.594,1.401
frozenset({'HighChol'}),frozenset({'HighBP'}),0.255,0.601,1.401


## 7. Decisão — regra de triagem e recomendações

**Regra de triagem (aplicável na atenção primária):**
SE (pressão alta = 1 E colesterol alto = 1 E faixa etária ≥ 9)
OU (IMC ≥ 30 E atividade física = 0)
ENTÃO encaminhar para rastreio prioritário de diabetes (glicemia de jejum + acompanhamento).

**Recomendações (3 prevenções priorizadas):**

1. Recomenda-se programa de atividade física para sedentários com IMC ≥ 30 — o grupo de maior prevalência na EDA.
2. Recomenda-se rastreio combinado pressão + colesterol em ≥ 55 anos (faixas 9–13), onde a prevalência dispara.
3. Recomenda-se acompanhamento de saúde geral (GenHlth ≥ 4) com avaliação de locomoção e dias de saúde ruim.

A amostra abaixo (200 linhas aleatórias, seed 42, sem estratificação, da base carregada —
reais quando `data/` presente, sintéticas caso contrário) replica o formato dos dados para auditoria.

In [10]:
sample = df.drop(columns=["cluster"], errors="ignore").sample(200, random_state=42)
outp = Path("..") / "docs" / "assets" / "data" / "sample_diabetes.csv"
outp.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(outp, index=False)
assert "cluster" not in sample.columns
print("sample REAL" if is_real else "sample SINTÉTICO", sample.shape, "bytes:", outp.stat().st_size)
display(HTML(df_to_html_table(sample, 10)))

sample REAL (200, 22) bytes: 18129


Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0.0,0.0,0.0,1.0,21.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,3.0,3.0,7.0,0.0,0.0,7.0,4.0,2.0
0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,13.0,6.0,6.0
0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,4.0,7.0
0.0,0.0,0.0,1.0,27.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,2.0,3.0,0.0,0.0,1.0,2.0,4.0,7.0
0.0,0.0,1.0,1.0,31.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,4.0,27.0,27.0,1.0,0.0,8.0,3.0,2.0
0.0,1.0,1.0,1.0,33.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,7.0,3.0,6.0
0.0,0.0,1.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0
0.0,0.0,1.0,1.0,27.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,10.0,4.0,8.0
0.0,1.0,1.0,1.0,25.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,8.0,6.0,8.0
0.0,0.0,0.0,1.0,33.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,8.0,5.0,8.0
